# 🇧🇷 Data Warehouse de Saúde Pública: COVID-19

Este projeto implementa um Data Warehouse (DW) utilizando dados públicos de COVID-19 (casos, mortes e vacinação) para fornecer uma plataforma analítica otimizada para insights epidemiológicos.

**Objetivo:** Transformar dados brutos em um **Modelo Dimensional (Schema Estrela)** que suporte consultas analíticas rápidas sobre a evolução da pandemia.

##  Domínio e Dataset

| Tópico | Detalhes |
| :--- | :--- |
| **Domínio** | Saúde Pública / Epidemiologia |
| **Dataset** | COVID-19 Data Repository (Our World in Data - OWID) |
| **Fonte** | `owid-covid-data.csv` |

##  Modelo Dimensional (Schema Estrela)

O DW é modelado como um Schema Estrela, sendo composto por:

| Tabela | Tipo | Descrição | Observação |
| :--- | :--- | :--- | :--- |
| `fact_covid_daily` | Tabela Fato | Contém as métricas diárias (novos casos, mortes, testes e vacinação). | Chaves estrangeiras para as dimensões. |
| `dim_date` | Dimensão | Detalhes temporais (ano, mês, dia da semana, etc.). | Criada via calendário gerado. |
| `dim_location` | Dimensão | Informações geográficas e socioeconômicas (país, continente, PIB). | **Implementada com SCD Type 2** para rastrear mudanças históricas. |
| `dim_health_indicators` | Dimensão | Indicadores de saúde e demográficos (população, idade média, etc.). | |

##  Stack Tecnológico

* **Banco de Dados:** [DuckDB](https://duckdb.org/) (Alta performance em OLAP, utilizado em modo de arquivo).
* **Linguagem ETL/Análise:** SQL (Padrão ANSI) e Python (para orquestração, pré-processamento e visualização).
* **Visualização:** Python (Plotly/Matplotlib).

##  Como Executar o Projeto (Guia do Notebook)

O notebook está estruturado para rodar o pipeline de dados em ordem sequencial.

### 1. Preparação (Instalação e Dados)

* Instala as bibliotecas Python necessárias (`duckdb`, `pandas`, `plotly`).
* Faz a leitura inicial do arquivo `owid-covid-data.csv`.

### 2. Pipeline ETL/ELT (Células 00 a 03)

| Etapa | Script (Mental) | Objetivo no Notebook |
| :--- | :--- | :--- |
| **00** | `00_staging.sql` | Cria Views temporárias no DuckDB a partir dos DataFrames. |
| **01** | `01_oltp.sql` | Normaliza e limpa os dados brutos (`oltp_location`, `oltp_daily_metrics`). |
| **02** | `02_dw_model.sql` | Cria as tabelas do Data Warehouse (`dim_*` e `fact_*`) vazias. |
| **03** | `03_etl_load.sql` | Popula dimensões (incluindo a lógica **SCD Type 2** na `dim_location`) e a Tabela Fato. |

### 3. Análise e Validação (Células 04 a 05)

* **Análise SQL:** Executa as 5 consultas analíticas obrigatórias (Temporal, Ranking, Agregação Multidimensional, Cohort e KPI).
* **Validação:** Executa a verificação de **Integridade Referencial** para garantir que a Tabela Fato não possui chaves órfãs (o resultado deve ser `0`).

### 4. Próximos Passos (A Fazer)

O código ETL e as análises estão funcionais. As próximas etapas do projeto são:

1.  **Visualizações:** Gerar e salvar os 4 gráficos obrigatórios (Linha, Barra, Heatmap/Dispersão, Dashboard).
2.  **Documentação Final:** Criar e organizar os arquivos `relatorio_tecnico.pdf`, `dicionario_dados.md` e o `diagrama_modelo_estrela.png` na pasta `docs/` para a entrega final no GitHub.

## 1. Introdução e Setup

Esta seção **configura o ambiente**, instala o **DuckDB** e baixa o *dataset* principal do **COVID-19**.

In [1]:
# 1. Instalação do DuckDB e Pandas
!pip install duckdb pandas requests

# 2. Importação de Bibliotecas
import duckdb
import pandas as pd
import os
import requests
from datetime import datetime

# Parâmetros de Conexão e Arquivo
DATA_URL = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
DATA_FILENAME = "owid-covid-data.csv"
DB_FILE = "covid_dw.duckdb"

# 3. Download do Arquivo (Apenas se não existir)
if not os.path.exists(DATA_FILENAME):
    print(f"Baixando {DATA_FILENAME}...")
    response = requests.get(DATA_URL)
    response.raise_for_status()
    with open(DATA_FILENAME, "wb") as f:
        f.write(response.content)
    print("Download concluído.")
else:
    print(f"O arquivo {DATA_FILENAME} já existe localmente.")

# 4. Conexão com o DuckDB (usando 'con' como objeto de conexão)
# O parâmetro read_only=False permite escrita no banco
con = duckdb.connect(database=DB_FILE, read_only=False)
print(f"\nConectado ao banco de dados: {DB_FILE}")

O arquivo owid-covid-data.csv já existe localmente.

Conectado ao banco de dados: covid_dw.duckdb


## 2.  ETL Passo 1: STAGING (`00_staging.sql`)

Cria uma **View Temporária** que lê o **CSV diretamente**, sem transformações, garantindo acesso rápido aos **dados brutos**.

In [2]:
# Script 00_staging.sql: Criação da View de Staging
sql_staging = f"""
-- 1. Criação da View Temporária para o CSV
CREATE OR REPLACE VIEW stg_covid_data AS
SELECT
    iso_code,
    continent,
    location AS location_name,
    date,
    population,
    new_cases,
    new_deaths,
    new_vaccinations,
    people_fully_vaccinated,
    icu_patients,
    gdp_per_capita,
    life_expectancy,
    hospital_beds_per_thousand,
    stringency_index
FROM
    read_csv_auto('{DATA_FILENAME}');
"""
con.execute(sql_staging)

# 2. Validação: Contagem de linhas
df_validation = con.execute("""
    SELECT
        COUNT(*) AS total_linhas_staging,
        COUNT(DISTINCT location_name) AS total_paises_unicos
    FROM
        stg_covid_data;
""").fetchdf()

print("✅ 00_staging.sql executado. View stg_covid_data criada.")
print("Validação do Staging:")
print(df_validation)

✅ 00_staging.sql executado. View stg_covid_data criada.
Validação do Staging:
   total_linhas_staging  total_paises_unicos
0                429435                  255


## 3.  ETL Passo 2: OLTP (`01_oltp.sql`)

O script **OLTP** tem como objetivo **normalizar e limpar os dados**, separando-os em **entidades lógicas** (`oltp_location` e `oltp_daily_metrics`).

In [3]:
# Script 01_oltp.sql: Normalização e Limpeza
sql_oltp = """
-- 1. DROP de tabelas OLTP (para idempotência)
DROP TABLE IF EXISTS oltp_location;
DROP TABLE IF EXISTS oltp_daily_metrics;

-- 2. Tabela OLTP de Localizações (Países)
-- Filtra agregados (continent = NULL) e remove duplicatas de atributos estáticos.
CREATE TABLE oltp_location AS
SELECT DISTINCT
    iso_code,
    continent,
    location_name,
    population,
    gdp_per_capita,
    life_expectancy
FROM
    stg_covid_data
WHERE
    iso_code IS NOT NULL
    AND continent IS NOT NULL
ORDER BY
    iso_code;

-- 3. Tabela OLTP de Métricas Diárias
-- Trata valores nulos (COALESCE) e garante tipo DATE.
CREATE TABLE oltp_daily_metrics AS
SELECT
    iso_code,
    date::DATE AS observation_date,
    COALESCE(new_cases, 0) AS new_cases,
    COALESCE(new_deaths, 0) AS new_deaths,
    COALESCE(new_vaccinations, 0) AS new_vaccinations,
    people_fully_vaccinated,
    icu_patients
FROM
    stg_covid_data
WHERE
    iso_code IS NOT NULL
    AND continent IS NOT NULL
    AND date IS NOT NULL
ORDER BY
    iso_code, observation_date;
"""
con.execute(sql_oltp)

# 4. Validação: Contagem de Linhas no OLTP
df_validation_oltp = con.execute("""
    SELECT 'oltp_location' AS tabela, COUNT(*) AS contagem FROM oltp_location
    UNION ALL
    SELECT 'oltp_daily_metrics' AS tabela, COUNT(*) AS contagem FROM oltp_daily_metrics;
""").fetchdf()

print("\n✅ 01_oltp.sql executado. Tabelas OLTP criadas.")
print("Validação do OLTP:")
print(df_validation_oltp)


✅ 01_oltp.sql executado. Tabelas OLTP criadas.
Validação do OLTP:
               tabela  contagem
0       oltp_location       243
1  oltp_daily_metrics    402910


## 4.  ETL Passo 3: DW Estrutura (`02_dw_model.sql`)

O objetivo desta etapa é criar o **Esquema Estrela** completo no DuckDB. O script é responsável por criar as **Tabelas Dimensão** (`dim_date`, `dim_location`, etc.) e a **Tabela Fato** (`fact_covid_daily`) com as respectivas **Chaves Substitutas**, garantindo que a estrutura do *Data Warehouse* esteja pronta para receber a carga de dados.

In [4]:
# Script 02_dw_model.sql: Criação da Estrutura do DW
sql_dw_model = """
-- 1. DROP das tabelas (garante idempotência)
DROP TABLE IF EXISTS fact_covid_daily;
DROP TABLE IF EXISTS dim_location;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_health_indicators;

-- 2. dim_date (Dimensão de Tempo)
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY, -- Formato YYYYMMDD
    date DATE NOT NULL,
    day_of_month INTEGER NOT NULL,
    month INTEGER NOT NULL,
    year INTEGER NOT NULL,
    day_of_week VARCHAR(10) NOT NULL,
    week_of_year INTEGER NOT NULL,
    is_weekend BOOLEAN NOT NULL
);

-- 3. dim_location (Dimensão de Localização - Implementa SCD Type 2)
CREATE TABLE dim_location (
    location_key INTEGER PRIMARY KEY,
    iso_code VARCHAR(10) NOT NULL,
    continent VARCHAR(50),
    location_name VARCHAR(100) NOT NULL,
    -- Atributos monitorados por SCD Type 2
    gdp_per_capita DECIMAL(18, 3),
    life_expectancy DECIMAL(18, 3),
    -- Colunas de controle SCD Type 2
    start_date DATE NOT NULL,
    end_date DATE,
    is_current BOOLEAN NOT NULL
);

-- 4. dim_health_indicators (Dimensão de Suporte - Usaremos dados agregados do Staging)
CREATE TABLE dim_health_indicators (
    indicator_key INTEGER PRIMARY KEY,
    iso_code VARCHAR(10) UNIQUE,
    hospital_beds_per_thousand DECIMAL(18, 3),
    stringency_index_avg DECIMAL(18, 3)
);

-- 5. fact_covid_daily (Tabela Fato - Granularidade: Dia x País)
CREATE TABLE fact_covid_daily (
    date_key INTEGER NOT NULL,
    location_key INTEGER NOT NULL,
    new_cases INTEGER,
    new_deaths INTEGER,
    new_vaccinations INTEGER,
    people_fully_vaccinated INTEGER,
    icu_patients INTEGER,
    population BIGINT,

    PRIMARY KEY (date_key, location_key),
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (location_key) REFERENCES dim_location(location_key)
);
"""
con.execute(sql_dw_model)

print("\n✅ 02_dw_model.sql executado. Estrutura do Data Warehouse criada.")


✅ 02_dw_model.sql executado. Estrutura do Data Warehouse criada.


## 5.  ETL Passo 4: Carga de Dados (`03_etl_load.sql`)

Esta etapa é a fase final do *pipeline* ETL/ELT e é responsável por **popular o Data Warehouse**. O script (`03_etl_load.sql`) inclui toda a lógica de negócio necessária para carregar as **tabelas dimensão** e a **tabela fato**.

### 5.1. Carga da dim_date

Iniciamos a carga populando a dimensão de tempo (`dim_date`), que é fundamental para todas as análises temporais.

In [5]:
# Obter o período de datas do OLTP
date_range = con.execute("SELECT MIN(observation_date), MAX(observation_date) FROM oltp_daily_metrics WHERE observation_date IS NOT NULL").fetchone()
MIN_DATE, MAX_DATE = date_range

sql_load_dim_date = f"""
-- Carga da dim_date com todas as datas do período
WITH date_series AS (
    SELECT
        UNNEST(GENERATE_SERIES(DATE '{MIN_DATE}', DATE '{MAX_DATE}', INTERVAL 1 DAY))::DATE AS dt
    -- A função UNNEST transforma o array de TIMESTAMPs (retornado pelo GENERATE_SERIES) em linhas individuais.
)
INSERT INTO dim_date
SELECT
    (CAST(strftime(dt, '%Y%m%d') AS INTEGER)) AS date_key,
    dt AS date,
    CAST(strftime(dt, '%d') AS INTEGER) AS day_of_month,
    CAST(strftime(dt, '%m') AS INTEGER) AS month,
    CAST(strftime(dt, '%Y') AS INTEGER) AS year,
    strftime(dt, '%a') AS day_of_week, -- Ex: 'Mon'
    CAST(strftime(dt, '%W') AS INTEGER) AS week_of_year,
    (CAST(strftime(dt, '%w') AS INTEGER) IN (0, 6)) AS is_weekend
FROM
    date_series
ON CONFLICT (date_key) DO NOTHING;
"""
con.execute(sql_load_dim_date)
print("\n✅ 03_etl_load.sql - dim_date carregada com sucesso!")


✅ 03_etl_load.sql - dim_date carregada com sucesso!


### 5.2. Carga das Dimensões Restantes (`dim_location` e `dim_health_indicators`)

Este bloco insere os dados iniciais dos países (incluindo a lógica **SCD2 simplificada** para o primeiro registro) e agrega os **indicadores de saúde**.

In [6]:
# Carga das dimensões `dim_location` e `dim_health_indicators`
sql_load_dimensions = """
-- 1. Carga da dim_location (SCD Type 2: Inserção Inicial)
-- Insere NOVOS PAÍSES com is_current = TRUE.
INSERT INTO dim_location (
    location_key, iso_code, continent, location_name,
    gdp_per_capita, life_expectancy,
    start_date, is_current
)
SELECT
    -- Gera uma chave substituta sequencial, garantindo que seja a PK
    (SELECT COALESCE(MAX(location_key), 0) FROM dim_location) + ROW_NUMBER() OVER (ORDER BY ol.iso_code) AS location_key_new,
    ol.iso_code,
    ol.continent,
    ol.location_name,
    ol.gdp_per_capita,
    ol.life_expectancy,
    (SELECT MIN(observation_date) FROM oltp_daily_metrics) AS start_date, -- Data de início igual à data mínima dos dados
    TRUE AS is_current
FROM
    oltp_location ol
LEFT JOIN
    dim_location dl ON ol.iso_code = dl.iso_code
WHERE
    dl.iso_code IS NULL; -- Apenas insere se o país não existe no DW

-- 2. Carga da dim_health_indicators
INSERT INTO dim_health_indicators (
    indicator_key, iso_code, hospital_beds_per_thousand, stringency_index_avg
)
SELECT
    ROW_NUMBER() OVER (ORDER BY stg.iso_code) AS indicator_key,
    stg.iso_code,
    FIRST(stg.hospital_beds_per_thousand) AS hospital_beds_per_thousand, -- Pega o primeiro valor disponível
    AVG(stg.stringency_index) AS stringency_index_avg
FROM
    stg_covid_data stg
WHERE
    stg.iso_code IS NOT NULL
GROUP BY
    stg.iso_code
ON CONFLICT (iso_code) DO UPDATE SET
    hospital_beds_per_thousand = EXCLUDED.hospital_beds_per_thousand,
    stringency_index_avg = EXCLUDED.stringency_index_avg;
"""
con.execute(sql_load_dimensions)
print("✅ 03_etl_load.sql - dim_location e dim_health_indicators carregadas.")

✅ 03_etl_load.sql - dim_location e dim_health_indicators carregadas.


### 5.3. Carga da `fact_covid_daily`

Este bloco carrega as **métricas diárias**, realizando a união (*JOIN*) com as **chaves substitutas** das dimensões (`dim_date`, `dim_location`, etc.).

In [7]:
# Carga da fact_covid_daily
sql_load_fact = """
INSERT INTO fact_covid_daily (
    date_key, location_key, new_cases, new_deaths, new_vaccinations, people_fully_vaccinated, icu_patients, population
)
SELECT
    dd.date_key,
    dl.location_key,
    odm.new_cases,
    odm.new_deaths,
    odm.new_vaccinations,
    odm.people_fully_vaccinated,
    odm.icu_patients,
    ol.population
FROM
    oltp_daily_metrics odm
JOIN
    dim_date dd ON odm.observation_date = dd.date
JOIN
    dim_location dl ON odm.iso_code = dl.iso_code AND dl.is_current = TRUE -- Usa a CHAVE CORRENTE do país
JOIN
    oltp_location ol ON odm.iso_code = ol.iso_code -- Junta para obter a população
ON CONFLICT (date_key, location_key) DO UPDATE SET
    new_cases = EXCLUDED.new_cases,
    new_deaths = EXCLUDED.new_deaths,
    new_vaccinations = EXCLUDED.new_vaccinations; -- Garante idempotência e atualização
"""
con.execute(sql_load_fact)

fact_count = con.execute('SELECT COUNT(*) FROM fact_covid_daily').fetchone()[0]
print(f"✅ 03_etl_load.sql - fact_covid_daily carregada com {fact_count} registros.")
print("\n**Checkpoint 2 (Pipeline ETL) CONCLUÍDO com sucesso!**")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ 03_etl_load.sql - fact_covid_daily carregada com 401502 registros.

**Checkpoint 2 (Pipeline ETL) CONCLUÍDO com sucesso!**


## 6.  Consultas Analíticas (`04_analytics.sql`)

Com o **Data Warehouse** populado, o próximo passo é a criação das **5 consultas analíticas obrigatórias** para responder às perguntas de negócio.

In [8]:
# Script 04_analytics.sql: Consultas Analíticas

# --- 1. Análise Temporal (Evolução de Casos/Mortes no Brasil por Mês/Ano) ---
sql_temporal = """
SELECT
    dd.year,
    dd.month,
    dl.location_name,
    SUM(f.new_cases) AS total_casos_mes,
    SUM(f.new_deaths) AS total_mortes_mes
FROM
    fact_covid_daily f
JOIN
    dim_date dd ON f.date_key = dd.date_key
JOIN
    dim_location dl ON f.location_key = dl.location_key
WHERE
    dl.location_name = 'Brazil'
GROUP BY
    dd.year, dd.month, dl.location_name
ORDER BY
    dd.year, dd.month;
"""
print("\n--- 1. Análise Temporal: Casos e Mortes no Brasil por Mês ---")
df_temporal = con.execute(sql_temporal).fetchdf()
print(df_temporal)

# --- 2. Ranking / TOP N (Top 10 Países por Taxa de Vacinação Completa) ---
# KPI: Pessoas totalmente vacinadas / População
sql_ranking = """
SELECT
    dl.location_name,
    dl.continent,
    MAX(f.people_fully_vaccinated) AS total_vacinados,
    MAX(f.population) AS populacao_total,
    (CAST(MAX(f.people_fully_vaccinated) AS DOUBLE) / MAX(f.population)) AS taxa_vacinacao_completa
FROM
    fact_covid_daily f
JOIN
    dim_location dl ON f.location_key = dl.location_key
WHERE
    dl.iso_code NOT IN ('OWID_WRL', 'OWID_EUR', 'OWID_ASI', 'OWID_AFR', 'OWID_OCE', 'OWID_NAM', 'OWID_SAM') -- Exclui agregados de continente/mundo
    AND f.people_fully_vaccinated IS NOT NULL
    AND f.population IS NOT NULL AND f.population > 0
GROUP BY
    dl.location_name, dl.continent
ORDER BY
    taxa_vacinacao_completa DESC
LIMIT 10;
"""
print("\n--- 2. Ranking / TOP 10: Países com Maior Taxa de Vacinação Completa ---")
df_ranking = con.execute(sql_ranking).fetchdf()
print(df_ranking)

# --- 3. Agregação Multidimensional (Taxa de Mortalidade por Continente e Expectativa de Vida) ---
# Taxa de Mortalidade Agregada = SUM(Mortes) / SUM(Casos)
sql_multidimensional = """
WITH summary AS (
    SELECT
        dl.continent,
        -- Cria grupos de Expectativa de Vida para a segunda dimensão
        CASE
            WHEN dl.life_expectancy < 70 THEN 'Baixa (<70 anos)'
            WHEN dl.life_expectancy BETWEEN 70 AND 80 THEN 'Média (70-80 anos)'
            ELSE 'Alta (>80 anos)'
        END AS grupo_expectativa_vida,
        SUM(f.new_deaths) AS total_mortes,
        SUM(f.new_cases) AS total_casos
    FROM
        fact_covid_daily f
    JOIN
        dim_location dl ON f.location_key = dl.location_key
    WHERE
        dl.continent IS NOT NULL
    GROUP BY
        dl.continent, grupo_expectativa_vida
)
SELECT
    continent,
    grupo_expectativa_vida,
    total_mortes,
    total_casos,
    (CAST(total_mortes AS DOUBLE) / total_casos) AS taxa_mortalidade_agregada
FROM
    summary
WHERE
    total_casos > 0
ORDER BY
    continent, grupo_expectativa_vida;
"""
print("\n--- 3. Agregação Multidimensional: Taxa de Mortalidade por Continente e Expectativa de Vida ---")
df_multidimensional = con.execute(sql_multidimensional).fetchdf()
print(df_multidimensional)

# --- 4. Análise de Cohort / Comportamento (Dias entre Início da Vacinação e Pico de Casos) ---
sql_cohort = """
WITH country_peaks AS (
    SELECT
        dl.iso_code,
        dl.location_name,
        -- Encontra o primeiro dia com vacinação > 100 (início da campanha)
        MIN(CASE WHEN f.new_vaccinations > 100 THEN dd.date END) AS data_inicio_vacinacao,
        -- Encontra a data do pico de novos casos
        FIRST(dd.date ORDER BY f.new_cases DESC) AS data_pico_casos
    FROM
        fact_covid_daily f
    JOIN
        dim_date dd ON f.date_key = dd.date_key
    JOIN
        dim_location dl ON f.location_key = dl.location_key
    WHERE
        dl.continent IS NOT NULL
        AND f.new_cases IS NOT NULL
    GROUP BY
        dl.iso_code, dl.location_name
)
SELECT
    location_name,
    data_inicio_vacinacao,
    data_pico_casos,
    (data_pico_casos - data_inicio_vacinacao) AS dias_ate_pico_apos_vacinacao
FROM
    country_peaks
WHERE
    data_inicio_vacinacao IS NOT NULL
    AND data_pico_casos IS NOT NULL
ORDER BY
    dias_ate_pico_apos_vacinacao DESC
LIMIT 10;
"""
print("\n--- 4. Análise de Cohort/Comportamento: Dias entre Início da Vacinação e Pico de Casos (Top 10) ---")
df_cohort = con.execute(sql_cohort).fetchdf()
print(df_cohort)

# --- 5. KPI (Indicador-Chave de Negócio: Taxa de Positividade Média por PIB per Capita) ---
sql_kpi = """
WITH country_kpi AS (
    -- Etapa 1: Calcula o KPI (taxa de casos) por país
    SELECT
        dl.location_name,
        dl.gdp_per_capita,
        SUM(f.new_cases) AS total_casos,
        MAX(f.population) AS populacao,
        (CAST(SUM(f.new_cases) AS DOUBLE) / MAX(f.population) * 100) AS taxa_casos_por_populacao
    FROM
        fact_covid_daily f
    JOIN
        dim_location dl ON f.location_key = dl.location_key
    WHERE
        dl.gdp_per_capita IS NOT NULL
        AND dl.continent IS NOT NULL
    GROUP BY
        dl.location_name, dl.gdp_per_capita
),
kpi_with_quartile AS (
    -- Etapa 2: Aplica a função de janela NTILE para definir o quartil
    SELECT
        taxa_casos_por_populacao,
        gdp_per_capita,
        NTILE(4) OVER (ORDER BY gdp_per_capita) AS quartil_pib_per_capita
    FROM country_kpi
)
-- Etapa 3: Agrupa pelo quartil já calculado e tira a média
SELECT
    quartil_pib_per_capita,
    AVG(taxa_casos_por_populacao) AS taxa_casos_media_percentual,
    MIN(gdp_per_capita) AS pib_min,
    MAX(gdp_per_capita) AS pib_max
FROM
    kpi_with_quartile
GROUP BY
    quartil_pib_per_capita
ORDER BY
    quartil_pib_per_capita;
"""
print("\n--- 5. KPI: Taxa de Casos Média por Quartil de PIB per Capita ---")
df_kpi = con.execute(sql_kpi).fetchdf()
print(df_kpi)

print("\n**Checkpoint 3 (Consultas Analíticas) CONCLUÍDO!**")


--- 1. Análise Temporal: Casos e Mortes no Brasil por Mês ---
    year  month location_name  total_casos_mes  total_mortes_mes
0   2020      1        Brazil              0.0               0.0
1   2020      2        Brazil              0.0               0.0
2   2020      3        Brazil           3417.0              92.0
3   2020      4        Brazil          49578.0            3578.0
4   2020      5        Brazil         412171.0           24208.0
5   2020      6        Brazil         809808.0           28083.0
6   2020      7        Brazil        1068392.0           29277.0
7   2020      8        Brazil        1461437.0           34266.0
8   2020      9        Brazil         884810.0           21033.0
9   2020     10        Brazil         664043.0           15934.0
10  2020     11        Brazil         884694.0           15503.0
11  2020     12        Brazil        1210210.0           18514.0
12  2021      1        Brazil        1669953.0           32178.0
13  2021      2        Braz

## 7. 📈 Visualizações e Insights (`05_visualizacoes.py`)

Esta etapa final do projeto envolve a **Geração dos gráficos** para a apresentação dos resultados. O código utilizará a biblioteca **Plotly** para criar visualizações interativas com base nos **DataFrames** gerados pelas consultas analíticas da etapa anterior.

In [15]:
# 1. Evolução Temporal
print("--- 1. Evolução Temporal ---")
fig_temporal.update_layout(xaxis_tickangle=-45)
# Corrigido: Substituído write_image() por show()
# fig_temporal.write_image("visualizacoes/1_evolucao_temporal.png")
fig_temporal.show()

# 2. Ranking de Vacinação
print("\n--- 2. Ranking de Vacinação ---")
fig_ranking.update_layout(xaxis_tickangle=-45)
# Corrigido: Substituído write_image() por show()
# fig_ranking.write_image("visualizacoes/2_ranking_vacinacao.png")
fig_ranking.show()

# 3. Mapa de Calor (Heatmap)
print("\n--- 3. Mapa de Calor (Heatmap) ---")
# Corrigido: Substituído write_image() por show()
# fig_heatmap.write_image("visualizacoes/3_mapa_calor.png")
fig_heatmap.show()

# 4. Dashboard (Combinação dos Gráficos)
print("\n--- 4. Dashboard (Combinação dos Gráficos) ---")
# A função write_html() salva um arquivo interativo e não causa o erro, por isso é mantida.
fig_dashboard.write_html("visualizacoes/4_dashboard_interativo.html")
print("Dashboard interativo salvo em: visualizacoes/4_dashboard_interativo.html")

# Adicionado fig.show() para exibir o dashboard também no output do notebook, para conveniência.
fig_dashboard.show()

--- 1. Evolução Temporal ---



--- 2. Ranking de Vacinação ---



--- 3. Mapa de Calor (Heatmap) ---



--- 4. Dashboard (Combinação dos Gráficos) ---
Dashboard interativo salvo em: visualizacoes/4_dashboard_interativo.html


## 8. Performance e Otimização (Bônus - `05_performance.sql`)

Esta seção é opcional, mas recomendada para ganhar os **pontos extras** do projeto. O foco é otimizar o desempenho de uma das *queries* analíticas mais pesadas.

### 8.1. Criação de Tabela Agregada (*Materialized View*)

A **Consulta 1** (Análise Temporal) é executada frequentemente e pode se beneficiar de uma agregação pré-calculada. O script criará uma **tabela agregada** (simulando uma *Materialized View*), pois a *query* envolve a junção de três tabelas e a agregação de milhões de linhas, o que é custoso em tempo de execução.

In [19]:
sql_validation = """
-- 1. Verifica registros da Fato sem correspondência em dim_date
SELECT
    'fcd_sem_dim_date' AS validacao,
    COUNT(*) AS count_erros
FROM
    fact_covid_daily f              -- Tabela FATO corrigida
LEFT JOIN
    dim_date dd ON f.date_key = dd.date_key
WHERE
    dd.date_key IS NULL

UNION ALL

-- 2. Verifica registros da Fato sem correspondência em dim_location
SELECT
    'fcd_sem_dim_location' AS validacao,
    COUNT(*) AS count_erros
FROM
    fact_covid_daily f              -- Tabela FATO corrigida
LEFT JOIN
    dim_location dl ON f.location_key = dl.location_key -- Tabela e CHAVE corrigidas
WHERE
    dl.location_key IS NULL;
"""

# Executa as consultas de validação e armazena o resultado em um DataFrame
df_validation_dw = con.execute(sql_validation).fetchdf()

print("--- Validação de Chaves Estrangeiras do Data Warehouse (DW) ---\n")
print(df_validation_dw)

# Validação final: os resultados devem ser 0 para garantir a integridade
if df_validation_dw['count_erros'].sum() == 0:
    print("\n✅ Validação de Integridade do DW CONCLUÍDA com SUCESSO! Todas as chaves foram mapeadas.")
else:
    print("\n❌ FALHA na Validação de Integridade do DW! Existem chaves não mapeadas. Verifique a etapa de carga (03_etl_load.sql).")

--- Validação de Chaves Estrangeiras do Data Warehouse (DW) ---

              validacao  count_erros
0      fcd_sem_dim_date            0
1  fcd_sem_dim_location            0

✅ Validação de Integridade do DW CONCLUÍDA com SUCESSO! Todas as chaves foram mapeadas.


### 8.2. Comparação de Performance da Consulta Otimizada

O **Data Warehouse** está modelado corretamente em **Esquema Estrela**, onde a *performance* das consultas complexas depende do tamanho da **Tabela Fato** e do número de *JOINs* necessários para obter as métricas.

#### 1. Contexto do Modelo de Dados (A Carga Original)

* **Tabela Fato Grande (Gargalo):** A tabela `fact_covid_daily` contém **401.502 registros**. Consultas que exigem *GROUP BY* e agregação desta tabela precisam escanear, processar e juntar centenas de milhares de linhas, o que é custoso em **I/O** (leitura de disco) e **CPU**.
* **Tabelas Dimensão Pequenas:** As dimensões (`dim_location` com 243 linhas e `dim_date`) são minúsculas.

#### 2. Análise da Consulta Original (Exemplo: Consulta 3)

A **consulta analítica de agregação multidimensional** (Consulta 3: Taxa de Mortalidade por Continente e Expectativa de Vida) é um exemplo de *query* que necessita escanear toda a Fato para agregar.

* **Processamento Original:** O DuckDB deve carregar e processar **401.502 linhas** da `fact_covid_daily` e, em seguida, juntá-las com a `dim_location` antes de realizar o *GROUP BY* e o cálculo do KPI.

#### 3. Cenário da Nova Consulta Otimizada (A Abordagem "Sem JOINs na Fato")

A otimização mais eficiente para consultas de agregação repetitiva é o uso de **Tabelas Agregadas** ou *Materialized Views*.

Se a "nova consulta (otimizada)" for realmente "simples, sem JOINs na Fato grande", isso implica que ela está consultando uma **Tabela Fato Agregada**.

| Tipo de Tabela | Número de Registros |
| :--- | :--- |
| **Tabela Fato Original** (`fact_covid_daily`) | 401.502 linhas |
| **Resultado Agregado** (Exemplo da Consulta 3) | 17 linhas |

In [22]:
sql_validation = """
-- 1. Verifica registros da Fato sem correspondência em dim_date
SELECT
    'fcd_sem_dim_date' AS validacao,
    COUNT(*) AS count_erros
FROM
    fact_covid_daily f              -- Tabela FATO correta
LEFT JOIN
    dim_date dd ON f.date_key = dd.date_key
WHERE
    dd.date_key IS NULL

UNION ALL

-- 2. Verifica registros da Fato sem correspondência em dim_location
SELECT
    'fcd_sem_dim_location' AS validacao,
    COUNT(*) AS count_erros
FROM
    fact_covid_daily f              -- Tabela FATO correta
LEFT JOIN
    dim_location dl ON f.location_key = dl.location_key -- Tabela e CHAVE corretas
WHERE
    dl.location_key IS NULL;
"""

# Executa as consultas de validação e armazena o resultado em um DataFrame
df_validation_dw = con.execute(sql_validation).fetchdf()

print("--- Validação de Chaves Estrangeiras do Data Warehouse (DW) ---\n")
print(df_validation_dw)

# Validação final: os resultados devem ser 0 para garantir a integridade
if df_validation_dw['count_erros'].sum() == 0:
    print("\n✅ Validação de Integridade do DW CONCLUÍDA com SUCESSO! Todas as chaves foram mapeadas.")
else:
    print("\n❌ FALHA na Validação de Integridade do DW! Existem chaves não mapeadas. Verifique a etapa de carga (03_etl_load.sql).")

--- Validação de Chaves Estrangeiras do Data Warehouse (DW) ---

              validacao  count_erros
0      fcd_sem_dim_date            0
1  fcd_sem_dim_location            0

✅ Validação de Integridade do DW CONCLUÍDA com SUCESSO! Todas as chaves foram mapeadas.


### 8.3. Validação da Integridade Referencial (Integridade de Chaves Estrangeiras)

O bloco de código e a execução de validação resumem-se à **Verificação da Integridade Referencial** (Integridade de Chaves Estrangeiras) do seu **Data Warehouse**.


O script SQL tem como objetivo primário garantir que **não há dados órfãos** na **tabela de fatos** (`fact_covid_daily`).

* **O que o SQL faz:** Utiliza dois `LEFT JOINs` (combinados por `UNION ALL`) para tentar conectar cada linha da **Tabela Fato** às suas dimensões (`dim_date` e `dim_location`).
* **Métrica de Sucesso:** Se o `LEFT JOIN` falha (ou seja, a chave substituta na dimensão é `NULL`), a linha é contada.
* **KPI:** O resultado final (`count_erros`) **deve ser zero**.

Esta é uma etapa crucial de **Qualidade de Dados** que confirma se a etapa de carga (ETL) foi bem-sucedida ao mapear todas as chaves, garantindo que as futuras consultas analíticas serão consistentes e não produzirão erros devido a ligações de dados ausentes.

#  Dicionário de Dados do Data Warehouse COVID-19

Este documento descreve as tabelas e as colunas que compõem o Data Warehouse (DW) de COVID-19, seguindo o modelo dimensional **Esquema Estrela**.

***

## 1. Tabela Fato: `fact_covid_daily`

Esta tabela armazena as **métricas diárias** (novos casos, mortes, vacinas, etc.) e está na granularidade de **Dia x Localização**.

| Coluna | Tipo | Descrição | Observação |
| :--- | :--- | :--- | :--- |
| **`date_key`** | `INTEGER` | Chave Estrangeira (FK) para a data da observação. | Liga à `dim_date`. Formato YYYYMMDD. |
| **`location_key`** | `INTEGER` | Chave Estrangeira (FK) que identifica a localização. | Liga à `dim_location`. |
| `new_cases` | `INTEGER` | Novos casos diários confirmados de COVID-19. | Medida Aditiva. |
| `new_deaths` | `INTEGER` | Novas mortes diárias reportadas devido à COVID-19. | Medida Aditiva. |
| `new_vaccinations` | `INTEGER` | Novas doses de vacinações aplicadas no dia. | Medida Aditiva. |
| `people_fully_vaccinated` | `INTEGER` | Contagem acumulada de pessoas totalmente vacinadas. | Não Aditiva (Snapshot). |
| `icu_patients` | `INTEGER` | Número de pacientes em Unidades de Terapia Intensiva (UTI) na data. | Não Aditiva (Snapshot). |
| `population` | `BIGINT` | População total da localização. | Não Aditiva. |

***

## 2. Tabela Dimensão: `dim_date`

Contém todos os atributos de tempo para análises por dia, mês, ano, semana, etc.

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| **`date_key`** | `INTEGER` | **Chave Primária (PK)**. Data no formato **YYYYMMDD**. |
| `date` | `DATE` | Data completa. |
| `day_of_month` | `INTEGER` | Dia numérico do mês (1 a 31). |
| `month` | `INTEGER` | Mês numérico do ano (1 a 12). |
| `year` | `INTEGER` | Ano. |
| `day_of_week` | `VARCHAR(10)` | Dia da semana (ex: 'Mon', 'Sat'). |
| `week_of_year` | `INTEGER` | Semana do ano (1 a 53). |
| `is_weekend` | `BOOLEAN` | Indica se a data é fim de semana (`TRUE`) ou dia útil (`FALSE`). |

***

## 3. Tabela Dimensão: `dim_location`

Armazena atributos geográficos e socioeconômicos. Implementada como **Slowly Changing Dimension (SCD) Type 2** para rastrear o histórico de `gdp_per_capita` e `life_expectancy`.

| Coluna | Tipo | Descrição | Observação |
| :--- | :--- | :--- | :--- |
| **`location_key`** | `INTEGER` | **Chave Primária (PK)** e Chave Substituta. | |
| `iso_code` | `VARCHAR(10)` | Código ISO padrão do país. | |
| `continent` | `VARCHAR(50)` | Continente. | |
| `location_name` | `VARCHAR(100)` | Nome da localização ou país. | |
| `gdp_per_capita` | `DECIMAL(18, 3)` | Produto Interno Bruto (PIB) per capita. | Atributo SCD Tipo 2. |
| `life_expectancy` | `DECIMAL(18, 3)` | Expectativa de vida ao nascer. | Atributo SCD Tipo 2. |
| `start_date` | `DATE` | Data de início da validade deste registro. | Controle SCD Type 2. |
| `end_date` | `DATE` | Data de fim da validade (NULL se for o registro atual). | Controle SCD Type 2. |
| `is_current` | `BOOLEAN` | Indica se este é o registro ativo (`TRUE`). | Controle SCD Type 2. |

***

## 4. Tabela Dimensão: `dim_health_indicators`

Contém indicadores de capacidade de saúde e rigidez de políticas governamentais.

| Coluna | Tipo | Descrição | Observação |
| :--- | :--- | :--- | :--- |
| **`indicator_key`** | `INTEGER` | **Chave Primária (PK)**. | |
| `iso_code` | `VARCHAR(10